# From DataFrame to Knowledge Graph: Music Edition

Three sources about bands and music in different formats with overlapping coverage.

**Sources:**
- `bands.csv`: a music encyclopedia (CSV)
- `vinyl_store.parquet`: a record store inventory (Parquet)
- `festival_lineups.csv`: festival schedules (CSV)

---
## Part 1: The data

In [ ]:
import polars as pl

bands    = pl.read_csv("data/bands.csv")
vinyl    = pl.read_parquet("data/vinyl_store.parquet")
lineups  = pl.read_csv("data/festival_lineups.csv")

bands

In [ ]:
vinyl

In [ ]:
lineups

Each source covers different aspects of the same bands:
- Band catalog has genre and country
- Vinyl store has albums and prices
- Festivals have stages and locations

All sources have different schemas and formats. How do you combine them?

---
## Part 2: Map to a knowledge graph

Initialise a new maplib Model and setting the namespace. 
- maplib Model _is_ your knowledge graph.
- namespace is the root path of your global unique identifiers (IRIs).

In [ ]:
from maplib import Model

m = Model()
ns = "http://example.org/music/"


### Source 1: Band encyclopedia (CSV)

We use OTTR templates as a "schema" for serialising data to a knowledge graph.
- Template variable names must correspond with DataFrame column headers. 
- Template body consits of instructions on how to serialise the input data to the knowledge graph.

In [ ]:
m.add_template("""
@prefix mu:<http://example.org/music/>.
@prefix xsd:<http://www.w3.org/2001/XMLSchema#>.

mu:Band [
    ottr:IRI ?band_iri,
    xsd:string ?name,
    xsd:string ?genre,
    xsd:long ?formed,
    xsd:string ?country
] :: {
    ottr:Triple(?band_iri, a,           mu:Band),
    ottr:Triple(?band_iri, mu:name,     ?name),
    ottr:Triple(?band_iri, mu:genre,    ?genre),
    ottr:Triple(?band_iri, mu:formed,   ?formed),
    ottr:Triple(?band_iri, mu:country,  ?country)
} .
""")

bands_df = bands.with_columns(
    (pl.lit(ns + "band/") + pl.col("name").str.replace_all(" ", "_")).alias("band_iri"),
).select(["band_iri", "name", "genre", "formed", "country"])

m.map(ns + "Band", bands_df)
print(f"Encyclopedia (CSV): {m.size()} triples")

### Source 2: Vinyl store (Parquet)

Albums link to band entities via an IRI. Because artist names match the encyclopedia, the store's albums connect directly to the same band entities.

In [ ]:
m.add_template("""
@prefix mu:<http://example.org/music/>.
@prefix xsd:<http://www.w3.org/2001/XMLSchema#>.

mu:Album [
    ottr:IRI ?album_iri,
    xsd:string ?title,
    ottr:IRI ?artist_iri,
    xsd:string ?artist_name,
    xsd:long ?year,
    xsd:double ?price,
    xsd:boolean ?in_stock
] :: {
    ottr:Triple(?album_iri, a,            mu:Album),
    ottr:Triple(?album_iri, mu:title,     ?title),
    ottr:Triple(?album_iri, mu:artist,    ?artist_iri),
    ottr:Triple(?album_iri, mu:year,      ?year),
    ottr:Triple(?album_iri, mu:price,     ?price),
    ottr:Triple(?album_iri, mu:inStock,   ?in_stock),
    ottr:Triple(?artist_iri, a,           mu:Band),
    ottr:Triple(?artist_iri, mu:name,     ?artist_name)
} .
""")

store_df = vinyl.with_columns(
    (pl.lit(ns + "album/") + pl.col("sku")).alias("album_iri"),
    (pl.lit(ns + "band/") + pl.col("artist").str.replace_all(" ", "_")).alias("artist_iri"),
    pl.col("artist").alias("artist_name"),
).rename({"album": "title", "price_usd": "price"}
).select(["album_iri", "title", "artist_iri", "artist_name", "year", "price", "in_stock"])

m.map(ns + "Album", store_df)
print(f"+ Vinyl store (Parquet): {m.size()} triples")

### Source 3: Festival lineups (CSV)

Same pattern: an OTTR template maps each lineup row to festival metadata and the band-at-festival link. Duplicate triples (same festival typed multiple times) are deduplicated by the graph.

In [ ]:
m.add_template("""
@prefix mu:<http://example.org/music/>.
@prefix xsd:<http://www.w3.org/2001/XMLSchema#>.

mu:FestivalLineup [
    ottr:IRI ?festival_iri,
    xsd:string ?festival,
    xsd:string ?location,
    ottr:IRI ?band_iri
] :: {
    ottr:Triple(?festival_iri, a,            mu:Festival),
    ottr:Triple(?festival_iri, mu:name,      ?festival),
    ottr:Triple(?festival_iri, mu:location,  ?location),
    ottr:Triple(?band_iri,     mu:playsAt,   ?festival_iri)
} .
""")

lineup_df = lineups.with_columns(
    (pl.lit(ns + "band/") + pl.col("artist").str.replace_all(" ", "_")).alias("band_iri"),
    (pl.lit(ns + "festival/") + pl.col("festival").str.replace_all(" ", "_")).alias("festival_iri"),
).select(["festival_iri", "festival", "location", "band_iri"])

m.map(ns + "FestivalLineup", lineup_df)
print(f"+ Festivals (CSV): {m.size()} triples")

Three sources, two formats, one graph.

---
## Part 3: Query across sources

### All bands with genre and country

In [ ]:
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?name ?genre ?country ?formed
    WHERE {
        ?b a mu:Band ;
           mu:name    ?name ;
           mu:genre   ?genre ;
           mu:country ?country ;
           mu:formed  ?formed .
    }
    ORDER BY ?name
""")

16 bands from the encyclopedia. The vinyl store mapped its albums to the same band IRIs, so genre, country, and album data are already connected on the same entity.

### Albums in stock for bands playing at Roadburn

In [ ]:
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?band_name ?album ?price
    WHERE {
        ?b mu:playsAt <http://example.org/music/festival/Roadburn> ;
           mu:name    ?band_name .
        ?a mu:artist  ?b ;
           mu:title   ?album ;
           mu:price   ?price ;
           mu:inStock true .
    }
    ORDER BY ?band_name ?album
""")

Roadburn has four bands, we get albums for all of them. The band names match across sources, so the IRIs connect automatically. No JOIN keys, no foreign keys, just shared identifiers.

### OPTIONAL: everything we know about each band

SPARQL OPTIONAL is like a LEFT JOIN. Get what's available, leave the rest as null.

In [ ]:
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?name ?genre ?country ?album ?price ?festival
    WHERE {
        ?b a mu:Band ;
           mu:name ?name .
        OPTIONAL { ?b mu:genre ?genre }
        OPTIONAL { ?b mu:country ?country }
        OPTIONAL { ?a mu:artist ?b ; mu:title ?album }
        OPTIONAL { ?a mu:price ?price }
        OPTIONAL { ?b mu:playsAt ?f . ?f mu:name ?festival }
    }
    ORDER BY ?name
""")

Every band that has albums also has genre and country. Bands without albums (Kraftwerk, NIN) still appear with their metadata. The OPTIONAL pattern works like a LEFT JOIN, no data is lost.

---
## Part 4: Enrich the graph

SPARQL CONSTRUCT derives new facts from patterns already in the graph.

### Genre vocabulary with SKOS

The genres are plain strings right now, and different sources spell them differently: "Post-Rock" vs "Post Rock", "Trip Hop" vs "Trip-Hop", "Progressive Rock" vs "Prog Rock". We can build a controlled vocabulary with canonical labels and alternative spellings, then swap the strings for Concept IRIs.

In [ ]:
# What genres do we have? (spot the variants)
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT DISTINCT ?genre WHERE { ?b a mu:Band ; mu:genre ?genre }
    ORDER BY ?genre
""")

In [ ]:
# Load the SKOS genre vocabulary (prefLabels, altLabels, broader relations)
m.read("data/rdf/genre_vocab.ttl")

print(f"Graph now has {m.size()} triples")

In [ ]:
# DELETE the string, INSERT the Concept IRI (matching on prefLabel OR altLabel)
m.update("""
    PREFIX mu: <http://example.org/music/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    DELETE { ?b mu:genre ?label }
    INSERT { ?b mu:genre ?concept }
    WHERE {
        ?b a mu:Band ; mu:genre ?label .
        FILTER(isLiteral(?label))
        ?concept a skos:Concept .
        { ?concept skos:prefLabel ?label }
        UNION
        { ?concept skos:altLabel ?label }
    }
""")

# Verify: "Post Rock" and "Post-Rock" now point to the same Concept
m.query("""
    PREFIX mu: <http://example.org/music/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?name ?genre_label
    WHERE {
        ?b a mu:Band ;
           mu:name  ?name ;
           mu:genre ?concept .
        ?concept skos:prefLabel ?genre_label .
    }
    ORDER BY ?genre_label ?name
""")

In [ ]:
# Save graph to Turtle — demo-extended.ipynb picks up from here
# Load the ontology (classes, properties, domain/range)
m.read("data/rdf/ontology.ttl")

p = {"mu" : "http://example.org/music/"}
m.write("data/rdf/music_graph.ttl", format="turtle", prefixes=p)
print(f"Graph written: {m.size()} triples → data/rdf/music_graph.ttl")

---
## Part 5: DataFrame back out

Every query result is a Polars DataFrame. Write it to Parquet, feed it to a pipeline, whatever.

In [ ]:
catalog = m.query("""
    PREFIX mu: <http://example.org/music/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

    SELECT ?name ?genre ?country ?formed ?era
    WHERE {
        ?b a mu:Band ;
           mu:name    ?name ;
           mu:genre   ?g ;
           mu:country ?country ;
           mu:formed  ?formed .
        ?g skos:prefLabel ?genre .
        OPTIONAL { ?b mu:era ?era }
    }
    ORDER BY ?formed
""")

catalog

In [ ]:
catalog.write_parquet("data/band_catalog.parquet")
print("Written to data/band_catalog.parquet")

### Query the schema itself

In SQL, the schema lives in a separate system (`information_schema`) that you query with a different mindset than your data. In a knowledge graph, the ontology _is_ data. Classes, properties, domains, and ranges are just more triples. One query language, one graph.

In [ ]:
m.query("""
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?class ?property ?range
    WHERE {
        ?cls a owl:Class .
        ?prop rdfs:domain ?cls ;
              rdfs:range  ?rng .
        BIND(REPLACE(STR(?cls),  ".*[/#]", "") AS ?class)
        BIND(REPLACE(STR(?prop), ".*[/#]", "") AS ?property)
        BIND(REPLACE(STR(?rng),  ".*[/#]", "") AS ?range)
    }
    ORDER BY ?class ?property
""")

---
## Part 6: Explore

Interactive graph browser. Try searching for "Deafheaven" or "Roadburn".

In [ ]:
m.insert("""
    PREFIX mu:   <http://example.org/music/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT { ?b rdfs:label ?name }
    WHERE     { ?b mu:name ?name }
""")

m.insert("""
    PREFIX mu:   <http://example.org/music/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT { ?a rdfs:label ?title }
    WHERE     { ?a mu:title ?title }
""")

server = m.explore(port=8765)

In [ ]:
server.stop()

---
## Recap

What we did:
1. Loaded three data sources (CSV, Parquet) into one graph
2. Mapped with OTTR templates
3. Queried across all sources with SPARQL: albums, prices, genres, and festivals connected through shared IRIs
4. Enriched: SKOS vocabulary for genres, CONSTRUCT for era classification
5. Saved the graph to Turtle for further enrichment
6. Got DataFrames back out, wrote to Parquet

**Next:** [demo-extended.ipynb](demo-extended.ipynb) picks up the saved graph and enriches it with external APIs (Wikidata, Discogs).

```
pip install maplib
```

- [maplib docs](https://datatreehouse.github.io/maplib/)
- [maplib on GitHub](https://github.com/DataTreehouse/maplib)
- [Why should you care about Knowledge Graphs?](https://veronahe.substack.com/p/data-engineer-why-should-you-care)
- [GitHub repo with runnable examples](https://github.com/veleda/data-engineering-to-knowledge-engineering)